# DLMS - AND, OR, NAND y NOR

Implementación DLMS con pesos aleatorios, diferentes valores de μ y criterio de convergencia.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


## 1. Datos de entrada

In [ ]:
X = np.array([
    [1, 0, 0],
    [1, 0, 1],
    [1, 1, 0],
    [1, 1, 1]
], dtype=float)


## 2. Salidas deseadas

In [ ]:
compuertas = {
    "AND": np.array([0, 0, 0, 1], dtype=float),
    "OR": np.array([0, 1, 1, 1], dtype=float),
    "NAND": np.array([1, 1, 1, 0], dtype=float),
    "NOR": np.array([1, 0, 0, 0], dtype=float)
}


## 3. Valores de μ

In [ ]:
mus = [0.10, 0.25, 0.50, 0.75]


## 4. Parámetros de entrenamiento

In [ ]:
epocas_maximas = 500
tolerancia = 0.001


## 5. Función DLMS

In [ ]:
def entrenar_dlms(X, d, mu, epocas_maximas, tolerancia, semilla=42):

    rng = np.random.default_rng(semilla)

    w = rng.uniform(
        low=-0.5,
        high=0.5,
        size=X.shape[1]
    )

    pesos_iniciales = w.copy()

    historial_error = []
    historial_error2 = []
    historial_salidas = []

    convergio = False
    epoca_convergencia = epocas_maximas

    for epoca in range(epocas_maximas):

        errores = []
        errores2 = []

        for i in range(len(X)):

            x = X[i]
            d_i = d[i]

            y = np.dot(w, x)

            error = d_i - y
            error2 = error ** 2

            errores.append(error)
            errores2.append(error2)

            # Actualización DLMS
            w = w + mu * error * x

        error_promedio = np.mean(np.abs(errores))
        mse = np.mean(errores2)

        historial_error.append(error_promedio)
        historial_error2.append(mse)

        salidas = np.dot(X, w)
        historial_salidas.append(salidas.copy())

        if mse <= tolerancia:
            convergio = True
            epoca_convergencia = epoca + 1
            break

    return {
        "pesos_iniciales": pesos_iniciales,
        "pesos_finales": w,
        "error": np.array(historial_error),
        "error2": np.array(historial_error2),
        "salidas": np.array(historial_salidas),
        "convergio": convergio,
        "epoca_convergencia": epoca_convergencia
    }


## 6. Entrenamiento

In [ ]:
resultados = {}

for indice_compuerta, (nombre, deseada) in enumerate(compuertas.items()):

    resultados[nombre] = {}

    for indice_mu, mu in enumerate(mus):

        semilla = 100 + indice_compuerta * 10 + indice_mu

        resultado = entrenar_dlms(
            X,
            deseada,
            mu,
            epocas_maximas,
            tolerancia,
            semilla
        )

        resultados[nombre][mu] = resultado


## 7. 16 gráficas de error y convergencia

In [ ]:
fig, axes = plt.subplots(
    4,
    4,
    figsize=(20, 16)
)

for fila, nombre in enumerate(compuertas):

    for columna, mu in enumerate(mus):

        ax = axes[fila, columna]

        resultado = resultados[nombre][mu]

        error = resultado["error"]
        error2 = resultado["error2"]

        epocas = np.arange(1, len(error) + 1)

        ax.plot(
            epocas,
            error,
            label="|Error| promedio"
        )

        ax.plot(
            epocas,
            error2,
            label="Error² / MSE"
        )

        ax.axhline(
            y=tolerancia,
            linestyle="--",
            label=f"ε = {tolerancia}"
        )

        if resultado["convergio"]:

            ec = resultado["epoca_convergencia"]

            ax.scatter(
                ec,
                error2[-1],
                s=70,
                zorder=5,
                label=f"Convergencia: {ec}"
            )

        ax.set_title(f"{nombre} | μ = {mu}")
        ax.set_xlabel("Época")
        ax.set_ylabel("Error")
        ax.grid(True)
        ax.legend(fontsize=7)

plt.suptitle(
    "DLMS - Error y convergencia",
    fontsize=18
)

plt.tight_layout()
plt.show()


## 8. 16 gráficas de aproximación al valor deseado

In [ ]:
fig, axes = plt.subplots(
    4,
    4,
    figsize=(20, 16)
)

nombres_patrones = [
    "(0,0)",
    "(0,1)",
    "(1,0)",
    "(1,1)"
]

for fila, (nombre, deseada) in enumerate(compuertas.items()):

    for columna, mu in enumerate(mus):

        ax = axes[fila, columna]

        resultado = resultados[nombre][mu]

        salidas = resultado["salidas"]

        epocas = np.arange(
            1,
            len(salidas) + 1
        )

        for patron in range(4):

            ax.plot(
                epocas,
                salidas[:, patron],
                label=(
                    f"{nombres_patrones[patron]}"
                    f" → {int(deseada[patron])}"
                )
            )

        ax.axhline(
            y=0,
            linestyle="--",
            alpha=0.25
        )

        ax.axhline(
            y=1,
            linestyle="--",
            alpha=0.25
        )

        if resultado["convergio"]:

            ax.axvline(
                x=resultado["epoca_convergencia"],
                linestyle=":",
                label=(
                    "Conv. "
                    f"{resultado['epoca_convergencia']}"
                )
            )

        ax.set_title(f"{nombre} | μ = {mu}")
        ax.set_xlabel("Época")
        ax.set_ylabel("Salida y")
        ax.grid(True)
        ax.legend(fontsize=7)

plt.suptitle(
    "DLMS - Aproximación a los valores deseados",
    fontsize=18
)

plt.tight_layout()
plt.show()


## 9. Resultados en consola

In [ ]:
print("\n")
print("=" * 85)
print("RESULTADOS DEL ENTRENAMIENTO DLMS")
print("=" * 85)

for nombre, deseada in compuertas.items():

    print("\n")
    print("=" * 70)
    print(f"COMPUERTA: {nombre}")
    print("=" * 70)

    for mu in mus:

        resultado = resultados[nombre][mu]

        pesos_iniciales = resultado["pesos_iniciales"]
        pesos_finales = resultado["pesos_finales"]

        salidas = resultado["salidas"][-1]

        salida_binaria = (salidas >= 0.5).astype(int)

        print(f"\nμ = {mu}")

        print(
            "Pesos iniciales aleatorios:",
            np.round(pesos_iniciales, 5)
        )

        print(
            "Pesos finales:",
            np.round(pesos_finales, 5)
        )

        print(
            "Salida deseada:",
            deseada.astype(int)
        )

        print(
            "Salida aproximada:",
            np.round(salidas, 5)
        )

        print(
            "Salida binaria:",
            salida_binaria
        )

        print(
            "Error final:",
            round(resultado["error"][-1], 8)
        )

        print(
            "Error² / MSE final:",
            round(resultado["error2"][-1], 8)
        )

        if resultado["convergio"]:

            print(
                "CONVERGIÓ en la época:",
                resultado["epoca_convergencia"]
            )

        else:

            print(
                "NO alcanzó la tolerancia de",
                tolerancia,
                "en",
                epocas_maximas,
                "épocas."
            )


## 10. Tabla resumen

In [ ]:
print("\n")
print("=" * 95)
print("TABLA RESUMEN")
print("=" * 95)

print(
    f"{'Compuerta':<12}"
    f"{'Mu':<10}"
    f"{'Épocas':<12}"
    f"{'Error':<15}"
    f"{'MSE':<15}"
    f"{'Convergencia':<15}"
)

print("-" * 95)

for nombre in compuertas:

    for mu in mus:

        resultado = resultados[nombre][mu]

        numero_epocas = len(resultado["error"])

        estado = (
            "Sí"
            if resultado["convergio"]
            else "No"
        )

        print(
            f"{nombre:<12}"
            f"{mu:<10}"
            f"{numero_epocas:<12}"
            f"{resultado['error'][-1]:<15.8f}"
            f"{resultado['error2'][-1]:<15.8f}"
            f"{estado:<15}"
        )
